# 🕷️ Spider-Man Incident Priority Classification

An end-to-end, reproducible machine learning workflow to predict incident response priority (`Low`, `Medium`, or `High`) from operational dispatch features.

---

### Table of Contents
1. [Setup and Global Configuration](#1-setup-and-global-configuration)
2. [Data Ingestion and Inspection](#2-data-ingestion-and-inspection)
3. [Structural Data Cleaning](#3-structural-data-cleaning)
4. [Train / Test Partitioning and Lockdown](#4-train--test-partitioning-and-lockdown)
5. [Exploratory Data Analysis (Training Portion Only)](#5-exploratory-data-analysis-training-portion-only)
6. [Data Leakage and Feature Audit](#6-data-leakage-and-feature-audit)
7. [Feature Engineering Assessment](#7-feature-engineering-assessment)
8. [Unified Preprocessing Pipeline](#8-unified-preprocessing-pipeline)
9. [Baseline Models](#9-baseline-models)
10. [Candidate Model Exploration and Hyperparameter Tuning](#10-candidate-model-exploration-and-hyperparameter-tuning)
11. [Fair Model Comparison and Selection Lock](#11-fair-model-comparison-and-selection-lock)
12. [Final Test-Set Evaluation](#12-final-test-set-evaluation)
13. [Post-Evaluation Error Analysis Facts](#13-post-evaluation-error-analysis-facts)
14. [Model Persistence and Verification](#14-model-persistence-and-verification)
15. [Writing Placeholders](#15-writing-placeholders)
16. [Final Evaluation Summary](#16-final-evaluation-summary)


## 1. Setup and Global Configuration

We import core Python scientific libraries and set global seeds so that every experiment and split yields identical results across machines.

In [1]:
import sys, os, json, hashlib, warnings, time, subprocess
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
import joblib
import sklearn

# Add local package root to Python path
sys.path.insert(0, str(Path(".").resolve()))
from src.config import (
    SEED, seed_everything, TARGET_NAME, TARGET_ORDER,
    ID_COLS, NUMERIC_COLS, CATEGORICAL_COLS,
    TEST_SIZE, N_SPLITS, N_ITER_SEARCH,
    PROJECT_ROOT, DATA_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR,
)

# Seed all random number generators
seed_everything(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
display(Markdown(f"Libraries imported. Random seed locked to `{SEED}`."))

Libraries imported. Random seed locked to `42`.

We verify runtime environment versions and confirm required report and model artifact directories are ready on disk.

In [2]:
# Ensure output directories exist
for folder in [FIGURES_DIR, MODELS_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Display environment metadata dynamically
display(Markdown(f"**Python Runtime**: `{sys.version.split()[0]}`"))
display(Markdown(f"**Pandas**: `{pd.__version__}` | **NumPy**: `{np.__version__}` | **Scikit-Learn**: `{sklearn.__version__}`"))
display(Markdown(f"**Global Deterministic SEED**: `{SEED}`"))
display(Markdown(f"**Target Class Ordering**: `{TARGET_ORDER}`"))

**Python Runtime**: `3.14.7`

**Pandas**: `3.1.0` | **NumPy**: `2.5.4` | **Scikit-Learn**: `1.9.1`

**Global Deterministic SEED**: `42`

**Target Class Ordering**: `['Low', 'Medium', 'High']`

## 2. Data Ingestion and Inspection

We safely load the raw incident dataset using robust encoding fallbacks and standardise column headers by stripping whitespace.

In [3]:
# Ingest dataset using progressive encoding fallback
raw_path = DATA_DIR / "spiderman_incident_priority_dataset.csv"
df_raw = None
for enc in ["utf-8", "utf-8-sig", "latin-1"]:
    try:
        df_raw = pd.read_csv(raw_path, encoding=enc)
        break
    except UnicodeDecodeError:
        continue

if df_raw is None:
    raise FileNotFoundError(f"Could not load dataset at {raw_path}")

# Clean column headers
df_raw.columns = df_raw.columns.str.strip()
display(Markdown(f"Loaded raw dataset with **{df_raw.shape[0]}** rows and **{df_raw.shape[1]}** columns."))

Loaded raw dataset with **1600** rows and **14** columns.

We validate the presence and nature of the target column to confirm this is a discrete classification task rather than regression.

In [4]:
# Confirm target column exists (case-insensitive lookup)
target_col = None
for col in df_raw.columns:
    if col.strip().lower().replace(" ", "_") == "response_priority":
        target_col = col
        break

if target_col is None:
    raise ValueError(f"Target column 'response_priority' missing. Available: {list(df_raw.columns)}")

if target_col != TARGET_NAME:
    df_raw = df_raw.rename(columns={target_col: TARGET_NAME})

# Confirm class count fits classification constraints
n_classes = df_raw[TARGET_NAME].nunique()
if pd.api.types.is_numeric_dtype(df_raw[TARGET_NAME]) and n_classes > 10:
    raise ValueError(f"Target has {n_classes} distinct numerical values (violates classification scope).")

display(Markdown(f"Target `{TARGET_NAME}` verified with **{n_classes}** distinct classes: `{list(df_raw[TARGET_NAME].unique())}`."))

Target `response_priority` verified with **3** distinct classes: `['Medium', 'Low', 'High']`.

We inspect data types, initial row samples, and summary statistics to understand the schema and values present in the raw table.

In [5]:
# Display summary table of schema properties
schema_summary = pd.DataFrame({
    "Data Type": df_raw.dtypes,
    "Missing Values": df_raw.isnull().sum(),
    "Missing (%)": (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    "Unique Values": df_raw.nunique(),
})
display(schema_summary)
display(Markdown(f"**Duplicate rows count**: {df_raw.duplicated().sum()}"))

,Data Type,Missing Values,Missing (%),Unique Values
incident_id,str,0,0.0,1600
villain,str,0,0.0,8
incident_type,str,0,0.0,7
location,str,0,0.0,7
weather,str,0,0.0,4
time_of_day,str,0,0.0,4
civilians_at_risk,int64,0,0.0,29
property_damage_level,int64,0,0.0,10
web_trap_strength,int64,0,0.0,10
spider_sense_alert,str,0,0.0,3


**Duplicate rows count**: 0

## 3. Structural Data Cleaning

We examine categorical frequency counts prior to cleaning to observe formatting inconsistencies and placeholder tokens.

In [6]:
# Inspect categorical distributions prior to cleaning
sample_cats = [c for c in ["weather", "time_of_day", "spider_sense_alert", "backup_status"] if c in df_raw.columns]
for col in sample_cats:
    print(f"--- Raw counts for {col} ---")
    print(df_raw[col].value_counts(dropna=False).head(5))
    print()

--- Raw counts for weather ---
weather
Clear    775
Rain     379
Storm    253
Fog      193
Name: count, dtype: int64

--- Raw counts for time_of_day ---
time_of_day
Night        431
Afternoon    421
Morning      376
Evening      372
Name: count, dtype: int64

--- Raw counts for spider_sense_alert ---
spider_sense_alert
Low         557
Moderate    528
High        515
Name: count, dtype: int64

--- Raw counts for backup_status ---
backup_status
Available      866
Delayed        464
Unavailable    270
Name: count, dtype: int64



We apply stateless structural cleaning from `src/cleaning.py` to normalise text casing and remove uninformative identifier and constant columns.

In [7]:
from src.cleaning import clean_dataframe

rows_start, cols_start = df_raw.shape
cleaning_records = []

# Apply stateless formatting and range validation
df_clean = clean_dataframe(df_raw)

# Remove ID-like columns that provide no generalisable signal
for id_col in ID_COLS:
    if id_col in df_clean.columns:
        df_clean = df_clean.drop(columns=[id_col])
        cleaning_records.append({"action": "drop_column", "column": id_col, "reason": "ID unique per row", "affected": rows_start})

# Remove uninformative constant features if any exist
const_cols = [c for c in df_clean.columns if df_clean[c].nunique(dropna=False) <= 1]
if const_cols:
    df_clean = df_clean.drop(columns=const_cols)
    for c in const_cols:
        cleaning_records.append({"action": "drop_column", "column": c, "reason": "constant zero variance", "affected": rows_start})

display(Markdown(f"Structural formatting applied. Columns retained: `{df_clean.shape[1]}`."))

Structural formatting applied. Columns retained: `13`.

We remove records with missing targets or exact duplicates, then export the completed cleaning audit log.

In [8]:
# Remove rows missing the supervised target label
missing_target_count = int(df_clean[TARGET_NAME].isnull().sum())
if missing_target_count > 0:
    df_clean = df_clean.dropna(subset=[TARGET_NAME])
    cleaning_records.append({"action": "drop_rows", "column": TARGET_NAME, "reason": "missing target", "affected": missing_target_count})

# Deduplicate identical records
dupes = int(df_clean.duplicated().sum())
if dupes > 0:
    df_clean = df_clean.drop_duplicates()
    cleaning_records.append({"action": "drop_duplicates", "column": "all", "reason": "exact duplicates", "affected": dupes})

# Record final audit log
rows_end, cols_end = df_clean.shape
cleaning_records.append({"action": "summary", "column": "all", "reason": f"({rows_start},{cols_start}) -> ({rows_end},{cols_end})", "affected": rows_start - rows_end})
cleaning_log_df = pd.DataFrame(cleaning_records)
cleaning_log_df.to_csv(REPORTS_DIR / "cleaning_log.csv", index=False)
display(cleaning_log_df)

,action,column,reason,affected
0,drop_column,incident_id,ID unique per row,1600
1,summary,all,"(1600,14) -> (1600,13)",0


## 4. Train / Test Partitioning and Lockdown

We partition the dataset into stratified training (80%) and holdout test (20%) subsets and save the partition hashes to enforce strict lockdown.

In [9]:
from sklearn.model_selection import train_test_split, StratifiedKFold

# Separate features from target
X_raw = df_clean.drop(columns=[TARGET_NAME])
y_raw = df_clean[TARGET_NAME]

# Perform stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=TEST_SIZE, stratify=y_raw, shuffle=True, random_state=SEED
)

# Calculate partition class distributions
train_dist = pd.DataFrame({"count": y_train.value_counts(), "pct": (y_train.value_counts(normalize=True) * 100).round(2)}).loc[TARGET_ORDER]
test_dist = pd.DataFrame({"count": y_test.value_counts(), "pct": (y_test.value_counts(normalize=True) * 100).round(2)}).loc[TARGET_ORDER]
display(Markdown(f"**Training Set**: {len(X_train)} samples | **Test Set**: {len(X_test)} samples"))
display(train_dist)

**Training Set**: 1280 samples | **Test Set**: 320 samples

,count,pct
response_priority,,
Low,180,14.06
Medium,900,70.31
High,200,15.62


We execute rigorous boundary checks confirming disjoint indices, absence of data leakage across the partition boundary, and save the locked test hash.

In [10]:
# Verify train and test index disjointness
overlap = set(X_train.index) & set(X_test.index)
assert len(overlap) == 0, f"Critical Data Leakage: {len(overlap)} overlapping indices!"

# Compute test-set checksum to guarantee immutable holdout data
test_concat = pd.concat([X_test, y_test], axis=1)
test_hash_val = hashlib.sha256(pd.util.hash_pandas_object(test_concat).values.tobytes()).hexdigest()

# Save split indices and hash to report directory
split_payload = {
    "train_indices": [int(i) for i in X_train.index],
    "test_indices": [int(i) for i in X_test.index],
    "test_hash_sha256": test_hash_val,
}
with open(REPORTS_DIR / "split_indices.json", "w") as f:
    json.dump(split_payload, f, indent=2)

# Set up stratified 5-fold cross validation on training data only
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
display(Markdown(f"✅ Partition verified. Holdout SHA-256: `{test_hash_val}`. Test set is now strictly locked."))

✅ Partition verified. Holdout SHA-256: `a44578e1c62448a48b57f240812280c0988ccdf8c9445dfec5af71b9fba77844`. Test set is now strictly locked.

## 5. Exploratory Data Analysis (Training Portion Only)

We visualise the target class distribution on the training set to quantify the class imbalance ratio in natural priority order.

In [11]:
# Visualise target class frequencies on training partition only
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
y_train_cat = y_train.astype(pd.CategoricalDtype(categories=TARGET_ORDER, ordered=True))
counts = y_train_cat.value_counts().sort_index()
colors = ["#2ecc71", "#f39c12", "#e74c3c"]

counts.plot(kind="bar", ax=axes[0], color=colors, edgecolor="black")
axes[0].set_title("Training Target Counts (Ordered)")
axes[0].set_ylabel("Incident Count")
axes[0].tick_params(axis="x", rotation=0)

pcts = (counts / counts.sum() * 100).round(1)
pcts.plot(kind="bar", ax=axes[1], color=colors, edgecolor="black")
axes[1].set_title("Training Target Proportions (%)")
axes[1].set_ylabel("Percentage (%)")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "target_distribution.png", dpi=150, bbox_inches="tight")
plt.close(fig)

imb_ratio = float(counts.max() / counts.min())
display(Markdown(f"Target imbalance ratio (dominant vs minority class): **{imb_ratio:.2f} : 1**"))

Target imbalance ratio (dominant vs minority class): **5.00 : 1**

We plot distributions and boxplots of numeric features broken down by target class to identify distinguishing ranges and outliers.

In [12]:
# Inspect numeric features grouped by target priority
train_view = X_train.copy()
train_view[TARGET_NAME] = y_train
active_num = [c for c in NUMERIC_COLS if c in X_train.columns]

fig, axes = plt.subplots(len(active_num), 2, figsize=(12, 3.2 * len(active_num)))
for i, col in enumerate(active_num):
    for cls, clr in zip(TARGET_ORDER, ["#2ecc71", "#f39c12", "#e74c3c"]):
        vals = train_view[train_view[TARGET_NAME] == cls][col].dropna()
        axes[i, 0].hist(vals, bins=15, alpha=0.5, label=cls, color=clr)
    axes[i, 0].set_title(f"{col} - Distribution by Priority")
    axes[i, 0].legend(fontsize=8)
    
    box_data = [train_view[train_view[TARGET_NAME] == cls][col].dropna() for cls in TARGET_ORDER]
    bp = axes[i, 1].boxplot(box_data, tick_labels=TARGET_ORDER, patch_artist=True)
    for box, clr in zip(bp["boxes"], ["#2ecc71", "#f39c12", "#e74c3c"]):
        box.set_facecolor(clr)
        box.set_alpha(0.6)
    axes[i, 1].set_title(f"{col} - Boxplot by Priority")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "numeric_distributions.png", dpi=150, bbox_inches="tight")
plt.close(fig)
display(Markdown("Generated numeric distribution histograms and class boxplots in `reports/figures/numeric_distributions.png`."))

Generated numeric distribution histograms and class boxplots in `reports/figures/numeric_distributions.png`.

We generate proportion bar charts for key categorical features across priority levels to assess differences in risk profile.

In [13]:
# Analyse categorical proportions across target priority levels
active_cat = [c for c in CATEGORICAL_COLS if c in X_train.columns]
for col in active_cat[:4]:
    top_categories = train_view[col].value_counts().head(8).index
    filtered = train_view[train_view[col].isin(top_categories)]
    ct = pd.crosstab(filtered[col], filtered[TARGET_NAME], normalize="index")
    ct = ct.reindex(columns=[c for c in TARGET_ORDER if c in ct.columns])
    
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ct.plot(kind="bar", stacked=True, ax=ax, color=["#2ecc71", "#f39c12", "#e74c3c"], edgecolor="black")
    ax.set_title(f"{col} Composition Across Priority Levels")
    ax.set_ylabel("Proportion")
    ax.legend(title="Priority", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f"cat_{col}_vs_target.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

display(Markdown("Categorical vs target proportion charts generated in `reports/figures/`."))

Categorical vs target proportion charts generated in `reports/figures/`.

We compute linear correlations among continuous features to detect multicollinearity among operational risk measures.

In [14]:
# Compute correlation matrix across numeric predictors
fig, ax = plt.subplots(figsize=(7, 5.5))
corr_matrix = train_view[active_num].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, cbar=True)
ax.set_title("Numeric Feature Correlation Matrix (Training Set)")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig)
display(Markdown("Feature correlation matrix saved to `reports/figures/correlation_heatmap.png`."))

Feature correlation matrix saved to `reports/figures/correlation_heatmap.png`.

We inspect missing value frequencies across all training predictors to confirm data completeness.

In [15]:
# Confirm absence of missing values on the training partition
missing_counts = X_train.isnull().sum()
fig, ax = plt.subplots(figsize=(8, 3.5))
missing_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Missing Values per Predictor (Training Set)")
ax.set_ylabel("Count of Missing Values")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "missingness.png", dpi=150, bbox_inches="tight")
plt.close(fig)

display(Markdown(f"Total missing values across all training predictors: **{missing_counts.sum()}**."))

Total missing values across all training predictors: **0**.

## 6. Data Leakage and Feature Audit

We evaluate predictive power per individual feature using Mutual Information and single-feature decision tree cross-validation to detect potential leakage.

In [16]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import OrdinalEncoder

# Prepare encoded training frame for mutual information computation
X_enc = X_train.copy()
for col in active_cat:
    X_enc[col] = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit_transform(X_enc[[col]].astype(str))
for col in active_num:
    X_enc[col] = X_enc[col].fillna(X_enc[col].median())

# Compute mutual information with target
target_enc = OrdinalEncoder().fit_transform(y_train.to_frame()).ravel()
discrete_mask = [c in active_cat for c in X_enc.columns]
mi_values = mutual_info_classif(X_enc, target_enc, discrete_features=discrete_mask, random_state=SEED)

# Benchmark single-feature decision tree macro-F1
single_f1_scores = {}
for col in X_enc.columns:
    dt = DecisionTreeClassifier(max_depth=3, random_state=SEED)
    cv_res = cross_validate(dt, X_enc[[col]], target_enc, cv=cv, scoring="f1_macro", n_jobs=-1)
    single_f1_scores[col] = float(cv_res["test_score"].mean())

display(Markdown("Single-feature diagnostic benchmarks calculated."))

Single-feature diagnostic benchmarks calculated.

We construct a structured leakage audit table to flag any post-decision columns or artificially predictive features.

In [17]:
# Build and export formal leakage audit register
audit_rows = []
for i, col in enumerate(X_enc.columns):
    f1_val = single_f1_scores[col]
    reasons, decision, confidence = [], "keep", "high"
    
    if f1_val > 0.90:
        reasons.append(f"Single-feature macro-F1={f1_val:.3f} > 0.90 (unrealistic single predictor)")
        decision, confidence = "drop", "high"
        
    if col == "response_delay_minutes":
        reasons.append("Semantic ambiguity: could represent observed post-dispatch elapsed delay")
        decision, confidence = "NEEDS HUMAN REVIEW", "medium"
        
    audit_rows.append({
        "column": col,
        "mutual_info": round(float(mi_values[i]), 4),
        "single_feature_f1_macro": round(f1_val, 4),
        "reason_flagged": "; ".join(reasons) if reasons else "none",
        "decision": decision,
        "confidence": confidence,
    })

audit_df = pd.DataFrame(audit_rows).sort_values("single_feature_f1_macro", ascending=False)
audit_df.to_csv(REPORTS_DIR / "leakage_audit.csv", index=False)
display(audit_df)

,column,mutual_info,single_feature_f1_macro,reason_flagged,decision,confidence
1,incident_type,0.1448,0.4545,none,keep,high
5,civilians_at_risk,0.0117,0.2801,none,keep,high
11,response_delay_minutes,0.0322,0.2779,Semantic ambiguity: could represent observed p...,NEEDS HUMAN REVIEW,medium
0,villain,0.0337,0.2752,none,keep,high
3,weather,0.0067,0.2752,none,keep,high
2,location,0.0535,0.2752,none,keep,high
6,property_damage_level,0.0313,0.2752,none,keep,high
4,time_of_day,0.0098,0.2752,none,keep,high
7,web_trap_strength,0.0319,0.2752,none,keep,high
8,spider_sense_alert,0.0154,0.2752,none,keep,high


We perform an ablation experiment on borderline features to quantify their impact on cross-validated macro-F1.

In [18]:
from sklearn.ensemble import RandomForestClassifier
from src.preprocessing import build_full_pipeline

# Baseline pipeline with all available predictors
pipe_all = build_full_pipeline(active_num, active_cat, RandomForestClassifier(n_estimators=200, random_state=SEED))
score_with = cross_validate(pipe_all, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)["test_score"].mean()

# Pipeline excluding borderline candidate response_delay_minutes
num_ablated = [c for c in active_num if c != "response_delay_minutes"]
pipe_ablated = build_full_pipeline(num_ablated, active_cat, RandomForestClassifier(n_estimators=200, random_state=SEED))
score_without = cross_validate(pipe_ablated, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)["test_score"].mean()

ablation_df = pd.DataFrame([{
    "feature_tested": "response_delay_minutes",
    "cv_macro_f1_with": round(score_with, 4),
    "cv_macro_f1_without": round(score_without, 4),
    "delta": round(score_with - score_without, 4),
    "status": "NEEDS HUMAN REVIEW",
}])
display(ablation_df)
display(Markdown("**Policy Decision**: Retained for baseline modelling as plausible estimated dispatch window, explicitly flagged for reviewer verification."))

,feature_tested,cv_macro_f1_with,cv_macro_f1_without,delta,status
0,response_delay_minutes,0.5754,0.5737,0.0016,NEEDS HUMAN REVIEW


**Policy Decision**: Retained for baseline modelling as plausible estimated dispatch window, explicitly flagged for reviewer verification.

## 7. Feature Engineering Assessment

We test domain-motivated interaction features and strictly evaluate whether they achieve the required +0.005 CV macro-F1 threshold.

In [19]:
from src.features import add_all_features, passthrough_features

# Evaluate model without engineered features
pipe_clean = build_full_pipeline(active_num, active_cat, RandomForestClassifier(n_estimators=200, random_state=SEED), feature_fn=passthrough_features)
score_plain = cross_validate(pipe_clean, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)["test_score"].mean()

# Evaluate model with domain engineered features
eng_num = active_num + ["risk_score", "damage_vs_trap"]
pipe_engineered = build_full_pipeline(eng_num, active_cat, RandomForestClassifier(n_estimators=200, random_state=SEED), feature_fn=add_all_features)
score_engineered = cross_validate(pipe_engineered, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)["test_score"].mean()

delta_fe = score_engineered - score_plain
keep_engineered = delta_fe >= 0.005

fe_summary = pd.DataFrame([{
    "transformations_tested": "risk_score, damage_vs_trap",
    "base_macro_f1": round(score_plain, 4),
    "engineered_macro_f1": round(score_engineered, 4),
    "delta": round(delta_fe, 4),
    "retained": keep_engineered,
}])
display(fe_summary)

# Define final active predictor sets based on evidence
FINAL_NUMERIC = eng_num if keep_engineered else active_num[:]
FINAL_CATEGORICAL = active_cat[:]
ACTIVE_FEAT_FN = add_all_features if keep_engineered else passthrough_features
display(Markdown(f"Feature engineering verdict: **{'Retained' if keep_engineered else 'Rejected'}** (Delta = {delta_fe:+.4f})."))

,transformations_tested,base_macro_f1,engineered_macro_f1,delta,retained
0,"risk_score, damage_vs_trap",0.5754,0.5702,-0.0052,False


Feature engineering verdict: **Rejected** (Delta = -0.0052).

## 8. Unified Preprocessing Pipeline

We construct and inspect the unified scikit-learn Pipeline architecture combining cleaning, transformation, and imputation.

In [20]:
from src.preprocessing import build_preprocessor

# Assemble reference end-to-end pipeline
sample_pipeline = build_full_pipeline(
    FINAL_NUMERIC, FINAL_CATEGORICAL,
    RandomForestClassifier(n_estimators=100, random_state=SEED),
    feature_fn=ACTIVE_FEAT_FN,
)

display(Markdown("### Pipeline Architecture Overview"))
for name, step in sample_pipeline.steps:
    display(Markdown(f"- **Step `{name}`**: `{type(step).__name__}`"))

display(Markdown("All transformations are embedded inside the pipeline to guarantee zero data leakage between train and inference."))

### Pipeline Architecture Overview

- **Step `clean`**: `FunctionTransformer`

- **Step `features`**: `FunctionTransformer`

- **Step `prep`**: `ColumnTransformer`

- **Step `model`**: `RandomForestClassifier`

All transformations are embedded inside the pipeline to guarantee zero data leakage between train and inference.

## 9. Baseline Models

We train and cross-validate two reference baselines: a majority-class DummyClassifier and an un-tuned LogisticRegression.

In [21]:
from src.models import get_baselines

baselines_dict = get_baselines()
baseline_metrics = {}

for b_name, b_estimator in baselines_dict.items():
    b_pipe = build_full_pipeline(FINAL_NUMERIC, FINAL_CATEGORICAL, b_estimator, feature_fn=ACTIVE_FEAT_FN)
    b_cv = cross_validate(
        b_pipe, X_train, y_train, cv=cv,
        scoring={"f1_macro": "f1_macro", "accuracy": "accuracy", "balanced_accuracy": "balanced_accuracy"},
        return_train_score=True, n_jobs=-1,
    )
    baseline_metrics[b_name] = {
        "cv_f1_macro_mean": float(b_cv["test_f1_macro"].mean()),
        "cv_f1_macro_std": float(b_cv["test_f1_macro"].std()),
        "cv_accuracy_mean": float(b_cv["test_accuracy"].mean()),
        "cv_balanced_accuracy_mean": float(b_cv["test_balanced_accuracy"].mean()),
        "train_f1_macro_mean": float(b_cv["train_f1_macro"].mean()),
    }

baseline_res_df = pd.DataFrame(baseline_metrics).T.round(4)
display(baseline_res_df)

baseline_target_f1 = max(r["cv_f1_macro_mean"] for r in baseline_metrics.values())
display(Markdown(f"**Baseline to Beat (CV Macro-F1)**: `{baseline_target_f1:.4f}`"))

,cv_f1_macro_mean,cv_f1_macro_std,cv_accuracy_mean,cv_balanced_accuracy_mean,train_f1_macro_mean
DummyMostFrequent,0.2752,0.0000,0.7031,0.3333,0.2752
LogisticRegression_default,0.8065,0.0199,0.8680,0.7763,0.8615


**Baseline to Beat (CV Macro-F1)**: `0.8065`

## 10. Candidate Model Exploration and Hyperparameter Tuning

We tune four diverse candidate families (Logistic Regression, Random Forest, HistGradientBoosting, KNN) using identical CV folds and macro-F1 scoring.

In [22]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from src.models import get_candidates

candidate_specs = get_candidates()
tuned_search_objects = {}
best_candidate_pipes = {}
best_params_recorded = {}

for model_name, (estimator, search_space, method) in candidate_specs.items():
    start_t = time.time()
    pipe = build_full_pipeline(FINAL_NUMERIC, FINAL_CATEGORICAL, estimator, feature_fn=ACTIVE_FEAT_FN)
    
    if method == "grid":
        search = GridSearchCV(pipe, search_space, cv=cv, scoring="f1_macro", n_jobs=-1, refit=True, return_train_score=True, error_score="raise")
    else:
        search = RandomizedSearchCV(pipe, search_space, n_iter=N_ITER_SEARCH, cv=cv, scoring="f1_macro", random_state=SEED, n_jobs=-1, refit=True, return_train_score=True, error_score="raise")
        
    search.fit(X_train, y_train)
    fit_duration = time.time() - start_t
    
    tuned_search_objects[model_name] = search
    best_candidate_pipes[model_name] = search.best_estimator_
    best_params_recorded[model_name] = search.best_params_
    
    # Save search diagnostics table
    pd.DataFrame(search.cv_results_).to_csv(REPORTS_DIR / f"tuning_{model_name}.csv", index=False)
    gap = search.cv_results_["mean_train_score"][search.best_index_] - search.best_score_
    display(Markdown(f"**{model_name}** | Best CV Macro-F1: `{search.best_score_:.4f}` | Train Gap: `{gap:.4f}` | Tuning Time: `{fit_duration:.1f}s`"))

**LogisticRegression** | Best CV Macro-F1: `0.8339` | Train Gap: `0.0487` | Tuning Time: `1.4s`

**RandomForest** | Best CV Macro-F1: `0.7231` | Train Gap: `0.2361` | Tuning Time: `14.7s`

**HistGradientBoosting** | Best CV Macro-F1: `0.7399` | Train Gap: `0.2041` | Tuning Time: `13.1s`

**KNN** | Best CV Macro-F1: `0.5406` | Train Gap: `0.4594` | Tuning Time: `3.3s`

We inspect parameter boundaries to verify that optimal hyperparameters do not sit unresolved on search range edges.

In [23]:
# Check if best hyperparameter configurations fall on search space extremes
edge_reports = []
for m_name, params in best_params_recorded.items():
    _, space, _ = candidate_specs[m_name]
    for param_key, param_val in params.items():
        if param_key in space and isinstance(space[param_key], list):
            options = space[param_key]
            if param_val in [options[0], options[-1]]:
                edge_reports.append(f"{m_name} -> `{param_key}` = {param_val} (boundary of {options})")

# Export best parameters to JSON
def json_serializable(val):
    if hasattr(val, "item"):
        return val.item()
    return str(val) if val is None else val

clean_params = {m: {k: json_serializable(v) for k, v in p.items()} for m, p in best_params_recorded.items()}
with open(REPORTS_DIR / "best_params.json", "w") as f:
    json.dump(clean_params, f, indent=2)

display(Markdown(f"Saved optimal parameter sets to `reports/best_params.json`. Edge warnings count: **{len(edge_reports)}**."))

Saved optimal parameter sets to `reports/best_params.json`. Edge warnings count: **9**.

## 11. Fair Model Comparison and Selection Lock

We assemble candidate models and baselines into a unified pool and configure a 15-fold repeated stratified validation scheme.

In [24]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, roc_auc_score

# Construct unified candidate registry
eval_pool = {}
for b_name, b_est in baselines_dict.items():
    eval_pool[b_name] = build_full_pipeline(FINAL_NUMERIC, FINAL_CATEGORICAL, b_est, feature_fn=ACTIVE_FEAT_FN)
for t_name, t_pipe in best_candidate_pipes.items():
    eval_pool[t_name] = t_pipe

fresh_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED + 1)
scoring_suite = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "recall_macro": make_scorer(recall_score, average="macro", zero_division=0),
    "f1_macro": "f1_macro",
    "f1_weighted": make_scorer(f1_score, average="weighted", zero_division=0),
    "roc_auc_ovr": make_scorer(roc_auc_score, multi_class="ovr", average="macro", response_method="predict_proba"),
}
display(Markdown(f"Unified pool configured with **{len(eval_pool)}** candidate pipelines across 15 validation folds."))

Unified pool configured with **6** candidate pipelines across 15 validation folds.

We cross-validate each model across repeated stratified folds and build a comprehensive comparison table.

In [25]:
fair_metrics = {}
for m_name, m_pipeline in eval_pool.items():
    t_start = time.time()
    cv_out = cross_validate(m_pipeline, X_train, y_train, cv=fresh_cv, scoring=scoring_suite, return_train_score=True, n_jobs=-1)
    fair_metrics[m_name] = {
        "cv_f1_macro_mean": float(cv_out["test_f1_macro"].mean()),
        "cv_f1_macro_std": float(cv_out["test_f1_macro"].std()),
        "cv_train_f1_macro": float(cv_out["train_f1_macro"].mean()),
        "train_val_gap": float(cv_out["train_f1_macro"].mean() - cv_out["test_f1_macro"].mean()),
        "cv_accuracy": float(cv_out["test_accuracy"].mean()),
        "cv_balanced_accuracy": float(cv_out["test_balanced_accuracy"].mean()),
        "cv_precision_macro": float(cv_out["test_precision_macro"].mean()),
        "cv_recall_macro": float(cv_out["test_recall_macro"].mean()),
        "cv_f1_weighted": float(cv_out["test_f1_weighted"].mean()),
        "cv_roc_auc_ovr": float(cv_out["test_roc_auc_ovr"].mean()),
        "fit_time_seconds": round(time.time() - t_start, 2),
    }

cv_comparison_df = pd.DataFrame(fair_metrics).T.sort_values("cv_f1_macro_mean", ascending=False)
cv_comparison_df.to_csv(REPORTS_DIR / "model_comparison_cv.csv")
display(cv_comparison_df.round(4))

,cv_f1_macro_mean,cv_f1_macro_std,cv_train_f1_macro,train_val_gap,cv_accuracy,cv_balanced_accuracy,cv_precision_macro,cv_recall_macro,cv_f1_weighted,cv_roc_auc_ovr,fit_time_seconds
LogisticRegression,0.8334,0.0176,0.8802,0.0469,0.8797,0.8246,0.8466,0.8246,0.8787,0.9662,0.31
LogisticRegression_default,0.8185,0.0192,0.8621,0.0436,0.8750,0.7885,0.8621,0.7885,0.8714,0.9640,0.30
HistGradientBoosting,0.7522,0.0379,0.9454,0.1933,0.8117,0.7697,0.7395,0.7697,0.8137,0.9220,0.70
RandomForest,0.7151,0.0290,0.9585,0.2434,0.7849,0.7296,0.7062,0.7296,0.7866,0.8982,1.66
KNN,0.5223,0.0404,1.0000,0.4777,0.7203,0.4912,0.6258,0.4912,0.6873,0.7406,0.29
DummyMostFrequent,0.2752,0.0000,0.2752,0.0000,0.7031,0.3333,0.2344,0.3333,0.5806,0.5000,0.25


We plot CV Macro-F1 performance across all models with error bars representing one cross-validation standard deviation.

In [26]:
# Plot model comparison with error bars
fig, ax = plt.subplots(figsize=(9, 4.5))
model_names_list = cv_comparison_df.index.tolist()
means = [fair_metrics[m]["cv_f1_macro_mean"] for m in model_names_list]
stds = [fair_metrics[m]["cv_f1_macro_std"] for m in model_names_list]

ax.bar(range(len(model_names_list)), means, yerr=stds, capsize=4, color="#3498db", edgecolor="black", alpha=0.85)
ax.set_xticks(range(len(model_names_list)))
ax.set_xticklabels(model_names_list, rotation=25, ha="right")
ax.set_ylabel("Macro-F1 Score")
ax.set_title("Repeated Stratified CV Macro-F1 (Mean ± 1 SD)")
ax.axhline(baseline_target_f1, color="red", linestyle="--", label=f"Baseline to Beat ({baseline_target_f1:.3f})")
ax.legend(loc="lower right")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "model_comparison_bar.png", dpi=150, bbox_inches="tight")
plt.close(fig)
display(Markdown("Generated CV comparison figure at `reports/figures/model_comparison_bar.png`."))

Generated CV comparison figure at `reports/figures/model_comparison_bar.png`.

We execute the pre-declared model selection rule, locking the chosen architecture before looking at any held-out test data.

In [27]:
# Pre-declared decision rule: maximum CV Macro-F1, tie-break for simplicity within 0.005
ranked_candidates = [m for m in cv_comparison_df.index if m not in baselines_dict]
top_candidate = ranked_candidates[0]
top_score = fair_metrics[top_candidate]["cv_f1_macro_mean"]

# Simplicity preference hierarchy
simplicity_order = ["LogisticRegression", "KNN", "RandomForest", "HistGradientBoosting"]
selected_model_name = top_candidate
for simple_m in simplicity_order:
    if simple_m in ranked_candidates:
        if abs(fair_metrics[simple_m]["cv_f1_macro_mean"] - top_score) <= 0.005:
            selected_model_name = simple_m
            break

beats_baselines_confirmed = fair_metrics[selected_model_name]["cv_f1_macro_mean"] > baseline_target_f1

# Write selection_locked.json to lock decision BEFORE touching test set
lock_payload = {
    "model": selected_model_name,
    "params": clean_params.get(selected_model_name, {}),
    "rule": "Highest repeated CV macro-F1; if within 0.005, choose simpler/faster architecture",
    "cv_f1_macro_mean": round(fair_metrics[selected_model_name]["cv_f1_macro_mean"], 4),
    "cv_f1_macro_std": round(fair_metrics[selected_model_name]["cv_f1_macro_std"], 4),
    "beats_baseline": beats_baselines_confirmed,
    "timestamp": datetime.now(timezone.utc).isoformat(),
}
with open(REPORTS_DIR / "selection_locked.json", "w") as f:
    json.dump(lock_payload, f, indent=2)

display(Markdown(f"🔒 **SELECTION LOCKED**: Selected **{selected_model_name}** at `{lock_payload['timestamp']}`."))

🔒 **SELECTION LOCKED**: Selected **LogisticRegression** at `2026-10-29T18:34:38.636416+00:00`.

We generate learning and validation curves on the training portion to diagnose model sample efficiency and sensitivity.

In [28]:
from sklearn.model_selection import learning_curve, validation_curve

chosen_candidate_pipe = eval_pool[selected_model_name]

# Learning curve across training fractions
train_sizes, train_scores, test_scores = learning_curve(
    chosen_candidate_pipe, X_train, y_train, cv=cv, scoring="f1_macro",
    train_sizes=np.linspace(0.15, 1.0, 8), random_state=SEED, n_jobs=-1
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_sizes, train_scores.mean(axis=1), "o-", color="#e74c3c", label="Training Score")
ax.plot(train_sizes, test_scores.mean(axis=1), "o-", color="#2980b9", label="Cross-Validation Score")
ax.set_title(f"Learning Curve ({selected_model_name})")
ax.set_xlabel("Training Samples")
ax.set_ylabel("Macro-F1 Score")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "learning_curve.png", dpi=150, bbox_inches="tight")
plt.close(fig)

display(Markdown("Learning curve diagnostics generated in `reports/figures/learning_curve.png`."))

Learning curve diagnostics generated in `reports/figures/learning_curve.png`.

## 12. Final Test-Set Evaluation

We verify the selection lock file exists, then evaluate all models on the held-out test set in this single official evaluation step.

In [29]:
# Guard assertion: final evaluation may ONLY execute after selection is locked
lock_file = REPORTS_DIR / "selection_locked.json"
assert lock_file.exists(), "Protocol Violation: reports/selection_locked.json does not exist!"

from src.evaluation import compute_all_metrics, adjacent_error_share

# Fit all pipelines on full training portion (tuned pipelines already refitted on X_train)
fitted_eval_suite = {}
for m_name, m_pipe in eval_pool.items():
    m_pipe.fit(X_train, y_train)
    fitted_eval_suite[m_name] = m_pipe

# Evaluate every pipeline on the holdout test set
test_evaluation_records = {}
for m_name, m_pipe in fitted_eval_suite.items():
    y_pred_holdout = m_pipe.predict(X_test)
    y_proba_holdout = m_pipe.predict_proba(X_test) if hasattr(m_pipe, "predict_proba") else None
    
    m_dict = compute_all_metrics(y_test.values, y_pred_holdout, y_proba_holdout, labels=TARGET_ORDER)
    m_dict["adjacent_error_share"] = adjacent_error_share(y_test.values, y_pred_holdout, order=TARGET_ORDER)
    test_evaluation_records[m_name] = m_dict

test_comparison_df = pd.DataFrame(test_evaluation_records).T.sort_values("f1_macro", ascending=False)
test_comparison_df.to_csv(REPORTS_DIR / "model_comparison_test.csv")
display(Markdown("### 🏆 FINAL TEST RESULTS"))
display(test_comparison_df.round(4))

### 🏆 FINAL TEST RESULTS

,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,mcc,quadratic_weighted_kappa,roc_auc_ovr_macro,log_loss,adjacent_error_share
LogisticRegression,0.8781,0.8385,0.8331,0.8385,0.8356,0.8782,0.7370,0.7957,0.9607,0.3023,1.0
LogisticRegression_default,0.8688,0.7785,0.8510,0.7785,0.8101,0.8649,0.7021,0.7585,0.9606,0.2948,1.0
HistGradientBoosting,0.8062,0.7704,0.7314,0.7704,0.7490,0.8094,0.6000,0.6927,0.9265,0.4269,1.0
RandomForest,0.7906,0.7111,0.7149,0.7111,0.7129,0.7905,0.5446,0.6454,0.8917,0.6234,1.0
KNN,0.7250,0.4807,0.6375,0.4807,0.5140,0.6869,0.2883,0.3610,0.7191,2.8269,1.0
DummyMostFrequent,0.7031,0.3333,0.2344,0.3333,0.2752,0.5806,0.0000,0.0000,0.5000,10.7005,1.0


We compute the detailed per-class classification report for the chosen model to inspect individual class precision, recall, and support.

In [30]:
from sklearn.metrics import classification_report

chosen_model_final = fitted_eval_suite[selected_model_name]
y_pred_final = chosen_model_final.predict(X_test)
y_proba_final = chosen_model_final.predict_proba(X_test)

class_report_dict = classification_report(y_test, y_pred_final, labels=TARGET_ORDER, output_dict=True)
class_report_df = pd.DataFrame(class_report_dict).T
display(Markdown(f"**Classification Breakdown for {selected_model_name} on Holdout Test Set:**"))
display(class_report_df.round(4))

**Classification Breakdown for LogisticRegression on Holdout Test Set:**

,precision,recall,f1-score,support
Low,0.8085,0.8444,0.8261,45.0000
Medium,0.9152,0.9111,0.9131,225.0000
High,0.7755,0.7600,0.7677,50.0000
accuracy,0.8781,0.8781,0.8781,0.8781
macro avg,0.8331,0.8385,0.8356,320.0000
weighted avg,0.8784,0.8781,0.8782,320.0000


We render raw count and row-normalised confusion matrices and One-vs-Rest ROC curves to assess prediction errors and ranking capability.

In [31]:
from src.evaluation import plot_confusion_matrices, plot_roc_curves

# Plot dual confusion matrix (raw counts + normalised proportions)
cm_fig = plot_confusion_matrices(y_test.values, y_pred_final, labels=TARGET_ORDER, save_path=str(FIGURES_DIR / "confusion_matrix.png"))
plt.close(cm_fig)

# Plot OvR ROC curves for each priority tier
roc_fig = plot_roc_curves(y_test.values, y_proba_final, classes=chosen_model_final.classes_, labels=TARGET_ORDER, save_path=str(FIGURES_DIR / "roc_curves.png"))
plt.close(roc_fig)

display(Markdown("Confusion matrix and ROC curve visuals rendered to `reports/figures/`."))

Confusion matrix and ROC curve visuals rendered to `reports/figures/`.

We compute non-parametric bootstrap 95% confidence intervals (1000 resamples) for macro-F1, accuracy, and balanced accuracy.

In [32]:
from src.evaluation import bootstrap_ci
from sklearn.metrics import accuracy_score, balanced_accuracy_score

metric_functions = {
    "f1_macro": lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0),
    "accuracy": accuracy_score,
    "balanced_accuracy": balanced_accuracy_score,
}

bootstrap_records = {}
for m_key, m_func in metric_functions.items():
    pt_est = float(m_func(y_test.values, y_pred_final))
    ci_low, ci_high = bootstrap_ci(y_test.values, y_pred_final, m_func, n_boot=1000, seed=SEED)
    bootstrap_records[m_key] = {"point": round(pt_est, 4), "ci_lower": round(ci_low, 4), "ci_upper": round(ci_high, 4)}

ci_display_df = pd.DataFrame(bootstrap_records).T
display(Markdown("**Bootstrap 95% Confidence Intervals (1000 Resamples, Seed 42):**"))
display(ci_display_df)

**Bootstrap 95% Confidence Intervals (1000 Resamples, Seed 42):**

,point,ci_lower,ci_upper
f1_macro,0.8356,0.7851,0.8801
accuracy,0.8781,0.8438,0.9125
balanced_accuracy,0.8385,0.7849,0.8901


We save the comprehensive final test metrics JSON and row-level test prediction logs to reports directory.

In [33]:
from sklearn.metrics import cohen_kappa_score

kappa_val = float(cohen_kappa_score(y_test, y_pred_final, labels=TARGET_ORDER, weights="quadratic"))
adj_err_val = float(adjacent_error_share(y_test.values, y_pred_final, order=TARGET_ORDER))

final_json_payload = {
    "chosen_model": {
        "name": selected_model_name,
        "metrics": test_evaluation_records[selected_model_name],
        "confidence_intervals": bootstrap_records,
        "quadratic_weighted_kappa": round(kappa_val, 4),
        "adjacent_error_share": round(adj_err_val, 4),
        "cv_f1_macro_mean": round(fair_metrics[selected_model_name]["cv_f1_macro_mean"], 4),
        "cv_f1_macro_std": round(fair_metrics[selected_model_name]["cv_f1_macro_std"], 4),
    },
    "baselines": {b: test_evaluation_records[b] for b in baselines_dict},
    "all_models": test_evaluation_records,
}

with open(REPORTS_DIR / "final_test_metrics.json", "w") as f:
    json.dump(final_json_payload, f, indent=2)

test_preds_df = pd.DataFrame({
    "row_index": X_test.index,
    "y_true": y_test.values,
    "y_pred": y_pred_final,
    "is_correct": (y_test.values == y_pred_final),
})
model_cls_list = list(chosen_model_final.classes_)
for cls_name in TARGET_ORDER:
    test_preds_df[f"proba_{cls_name}"] = y_proba_final[:, model_cls_list.index(cls_name)]
test_preds_df.to_csv(REPORTS_DIR / "test_predictions.csv", index=False)

display(Markdown("✅ Test metrics and prediction logs successfully saved to `reports/`."))

✅ Test metrics and prediction logs successfully saved to `reports/`.

## 13. Post-Evaluation Error Analysis Facts

We compute objective confusion pairs, per-class recall rankings, prediction confidences, and the 25 most confident error cases.

In [34]:
from sklearn.metrics import confusion_matrix

cm_arr = confusion_matrix(y_test, y_pred_final, labels=TARGET_ORDER)
pairs = []
for i, true_l in enumerate(TARGET_ORDER):
    for j, pred_l in enumerate(TARGET_ORDER):
        if i != j and cm_arr[i, j] > 0:
            pairs.append({"true": true_l, "pred": pred_l, "count": int(cm_arr[i, j]), "pct": round(cm_arr[i, j] / cm_arr[i].sum() * 100, 1)})
pairs.sort(key=lambda x: x["count"], reverse=True)

# Rank class recalls
recalls = {cls: round(float((y_pred_final[y_test == cls] == cls).sum() / (y_test == cls).sum()), 4) for cls in TARGET_ORDER}
confidences = y_proba_final.max(axis=1)
correct_flag = (y_test.values == y_pred_final)

# Isolate top 25 high-confidence errors
err_mask = ~correct_flag
top25_err_df = X_test[err_mask].copy()
top25_err_df["true_label"] = y_test.values[err_mask]
top25_err_df["pred_label"] = y_pred_final[err_mask]
top25_err_df["confidence"] = confidences[err_mask]
top25_err_df = top25_err_df.sort_values("confidence", ascending=False).head(25)

display(Markdown(f"Identified **{len(pairs)}** distinct error pairings. Most confident errors isolated."))

Identified **4** distinct error pairings. Most confident errors isolated.

We compute permutation feature importances on the test set and export all factual error findings to `reports/error_analysis_facts.md`.

In [35]:
from sklearn.inspection import permutation_importance

perm_res = permutation_importance(chosen_model_final, X_test, y_test, scoring="f1_macro", n_repeats=10, random_state=SEED, n_jobs=-1)
perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm_res.importances_mean,
    "importance_std": perm_res.importances_std,
}).sort_values("importance_mean", ascending=False).head(15)

# Render permutation importance bar chart
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(len(perm_df)), perm_df["importance_mean"], xerr=perm_df["importance_std"], color="#16a085", capsize=3)
ax.set_yticks(range(len(perm_df)))
ax.set_yticklabels(perm_df["feature"])
ax.invert_yaxis()
ax.set_xlabel("Mean Test Macro-F1 Drop")
ax.set_title("Test Set Permutation Importance (Top 15 Features)")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "permutation_importance.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# Write comprehensive markdown facts file (strictly factual, zero interpretation)
md_facts = ["# Error Analysis Factual Summary\n\n## Top Confusion Pairs\n"]
for p in pairs[:5]:
    md_facts.append(f"- **{p['true']} -> {p['pred']}**: {p['count']} instances ({p['pct']}% of true {p['true']})")
md_facts.append("\n## Recall Per Class (Ranked)\n")
for cls, r_val in sorted(recalls.items(), key=lambda x: x[1], reverse=True):
    md_facts.append(f"- **{cls}**: {r_val:.4f}")
md_facts.append(f"\n## Prediction Confidence\n- Mean Confidence (Correct Predictions): `{confidences[correct_flag].mean():.4f}`\n- Mean Confidence (Errors): `{confidences[~correct_flag].mean():.4f}`\n")
md_facts.append("\n## Permutation Feature Importance (Top Features)\n")
md_facts.append("| Feature | Mean Macro-F1 Drop | Std |\n|---|---|---|")
for _, r in perm_df.iterrows():
    md_facts.append(f"| {r['feature']} | {r['importance_mean']:.4f} | {r['importance_std']:.4f} |")

with open(REPORTS_DIR / "error_analysis_facts.md", "w", encoding="utf-8") as f:
    f.write("\n".join(md_facts))
display(Markdown("Exported `reports/error_analysis_facts.md`."))

Exported `reports/error_analysis_facts.md`.

We compile the objective numerical facts required for external justification and save them to `reports/facts_for_justification.md`.

In [36]:
# Generate factual justification briefing
class_freq_str = str(dict(y_raw.value_counts()))
facts_brief = [
    "# Dataset and Modelling Facts for Justification\n",
    f"- Total Dataset Size (Cleaned): `{len(df_clean)}` rows × `{df_clean.shape[1]}` columns",
    f"- Active Predictors: `{len(FINAL_NUMERIC)}` numeric, `{len(FINAL_CATEGORICAL)}` categorical",
    f"- Class Frequencies: `{class_freq_str}`",
    f"- Class Imbalance Ratio: `{float(y_raw.value_counts().max() / y_raw.value_counts().min()):.2f}`",
    f"- Baseline CV Macro-F1: `{baseline_target_f1:.4f}`",
    f"- Selected Architecture: `{selected_model_name}`",
    f"- Chosen Model CV Macro-F1: `{fair_metrics[selected_model_name]['cv_f1_macro_mean']:.4f} ± {fair_metrics[selected_model_name]['cv_f1_macro_std']:.4f}`",
    f"- Chosen Model Test Macro-F1: `{test_evaluation_records[selected_model_name]['f1_macro']:.4f}`",
    f"- Chosen Model Test Accuracy: `{test_evaluation_records[selected_model_name]['accuracy']:.4f}`",
    "- Leakage Decisions: `response_delay_minutes` retained as plausible dispatch estimate, pending operational signoff.",
]
with open(REPORTS_DIR / "facts_for_justification.md", "w", encoding="utf-8") as f:
    f.write("\n".join(facts_brief))
display(Markdown("Exported `reports/facts_for_justification.md`."))


Exported `reports/facts_for_justification.md`.

## 14. Model Persistence and Verification

We serialise the complete trained pipeline directly to disk with compression.

In [37]:
# Save complete production pipeline
joblib.dump(chosen_model_final, MODELS_DIR / "best_model.joblib", compress=3)
display(Markdown("✅ Trained pipeline serialised to `models/best_model.joblib` (compress=3)."))

✅ Trained pipeline serialised to `models/best_model.joblib` (compress=3).

We construct and export the comprehensive model metadata file documenting training schemas and performance.

In [38]:
# Build feature schema registry listing input column properties
schema_catalog = {}
for col in X_train.columns:
    if col in FINAL_NUMERIC:
        schema_catalog[col] = {
            "kind": "numeric",
            "min": float(X_train[col].min()),
            "median": float(X_train[col].median()),
            "max": float(X_train[col].max()),
        }
    elif col in FINAL_CATEGORICAL:
        vc = X_train[col].value_counts()
        schema_catalog[col] = {
            "kind": "categorical",
            "categories": vc.index.tolist(),
            "frequencies": [int(v) for v in vc.values],
        }

metadata_payload = {
    "model_name": selected_model_name,
    "best_params": clean_params.get(selected_model_name, {}),
    "class_labels": TARGET_ORDER,
    "target_name": TARGET_NAME,
    "input_schema": schema_catalog,
    "python_version": sys.version.split()[0],
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "sklearn_version": sklearn.__version__,
    "seed": SEED,
    "dataset_sha256": hashlib.sha256(open(DATA_DIR / "spiderman_incident_priority_dataset.csv", "rb").read()).hexdigest(),
    "final_test_metrics": test_evaluation_records[selected_model_name],
}
with open(MODELS_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata_payload, f, indent=2)

display(Markdown("✅ Model metadata exported to `models/model_metadata.json`."))

✅ Model metadata exported to `models/model_metadata.json`.

We verify round-trip consistency by loading the saved model in an isolated Python process and checking prediction parity on training samples.

In [39]:
# Verify round-trip predictions in an isolated process
verify_cmd = [
    sys.executable, "-c",
    "import joblib, pandas as pd, json, sys, numpy as np; "
    "model = joblib.load('models/best_model.joblib'); "
    "X_sample = pd.read_csv('data/spiderman_incident_priority_dataset.csv').drop(columns=['response_priority']).head(5); "
    "preds = model.predict(X_sample).tolist(); "
    "print(json.dumps(preds))"
]
proc = subprocess.run(verify_cmd, capture_output=True, text=True, check=True)
isolated_preds = json.loads(proc.stdout.strip())
in_memory_preds = chosen_model_final.predict(df_clean.drop(columns=[TARGET_NAME]).head(5)).tolist()

assert isolated_preds == in_memory_preds, f"Round-trip mismatch: {isolated_preds} != {in_memory_preds}"
display(Markdown(f"✅ Round-trip validation passed: predictions match identically across isolated environments: `{isolated_preds}`."))

✅ Round-trip validation passed: predictions match identically across isolated environments: `['Medium', 'Medium', 'Medium', 'Medium', 'Medium']`.

## 15. Writing Placeholders

The sections below are reserved for the author to draft qualitative analysis. Guiding questions are provided.

## ✍️ Model justification: WRITE THIS YOURSELF

**Guiding questions:**
- What does the data look like (size, feature types, imbalance, missing values, non-linear or interaction structure)?
- What did I compare against and what were the scores?
- Why does the chosen model fit this data better than the others?
- What are its limits?

*(Refer to `reports/facts_for_justification.md` for verified figures.)*


## ✍️ Overfitting, underfitting and error discussion: WRITE THIS YOURSELF

**Guiding questions:**
- What do the train-vs-validation gaps and the learning curve show?
- Which classes get confused and why might that be?
- Is the test score consistent with CV?

*(Refer to `reports/error_analysis_facts.md` for verified counts and pairs.)*


## ✍️ Why these metrics: REWRITE IN YOUR OWN WORDS

**DRAFT** (definitions only — rewrite in your own words):

- **Accuracy**: The overall fraction of correct predictions across all cases. Because Medium-priority incidents represent approximately 70% of the dataset, a naive model predicting only Medium would achieve ~70% accuracy while failing completely on high-risk emergencies.
- **Precision (macro-averaged)**: The proportion of positive predictions that are truly correct, calculated per priority tier and averaged with equal weighting across classes.
- **Recall (macro-averaged)**: The proportion of actual incidents in each priority tier correctly identified, averaged equally across classes to avoid masking severe minority classes.
- **Macro-F1 Score**: The unweighted harmonic mean of macro precision and macro recall. It assigns equal importance to detecting Low, Medium, and High incidents, penalising models that ignore the rarer High or Low classes.
- **ROC-AUC (One-vs-Rest, Macro)**: Measures the average pairwise ranking discrimination capability across each individual class boundary.
- **Why macro-averaging?**: In incident emergency response, failing to dispatch to a critical High incident is catastrophic, so treating each priority level with equal evaluation weight is mandatory.


## 16. Final Evaluation Summary

We load and print the verified test metrics summary directly from `reports/final_test_metrics.json`.

In [40]:
# Load verified metrics from final JSON
with open(REPORTS_DIR / "final_test_metrics.json") as f:
    final_data = json.load(f)

chosen_info = final_data["chosen_model"]
display(Markdown(f"### Selected Model: **{chosen_info['name']}**"))
display(Markdown(f"- **Test Macro-F1**: `{chosen_info['metrics']['f1_macro']:.4f}` (95% CI: `[{chosen_info['confidence_intervals']['f1_macro']['ci_lower']:.4f}, {chosen_info['confidence_intervals']['f1_macro']['ci_upper']:.4f}]`)"))
display(Markdown(f"- **Test Accuracy**: `{chosen_info['metrics']['accuracy']:.4f}` (95% CI: `[{chosen_info['confidence_intervals']['accuracy']['ci_lower']:.4f}, {chosen_info['confidence_intervals']['accuracy']['ci_upper']:.4f}]`)"))
display(Markdown(f"- **Quadratic-Weighted Kappa**: `{chosen_info['quadratic_weighted_kappa']:.4f}`"))
display(Markdown(f"- **Adjacent Error Share**: `{chosen_info['adjacent_error_share']:.4f}`"))

# Comparative table
summary_table = pd.DataFrame(final_data["all_models"]).T[["accuracy", "balanced_accuracy", "f1_macro", "f1_weighted", "mcc"]]
summary_table["selected"] = ["⭐" if idx == chosen_info["name"] else "" for idx in summary_table.index]
display(summary_table.round(4))

### Selected Model: **LogisticRegression**

- **Test Macro-F1**: `0.8356` (95% CI: `[0.7851, 0.8801]`)

- **Test Accuracy**: `0.8781` (95% CI: `[0.8438, 0.9125]`)

- **Quadratic-Weighted Kappa**: `0.7957`

- **Adjacent Error Share**: `1.0000`

,accuracy,balanced_accuracy,f1_macro,f1_weighted,mcc,selected
DummyMostFrequent,0.7031,0.3333,0.2752,0.5806,0.0000,
LogisticRegression_default,0.8688,0.7785,0.8101,0.8649,0.7021,
LogisticRegression,0.8781,0.8385,0.8356,0.8782,0.7370,⭐
RandomForest,0.7906,0.7111,0.7129,0.7905,0.5446,
HistGradientBoosting,0.8062,0.7704,0.7490,0.8094,0.6000,
KNN,0.7250,0.4807,0.5140,0.6869,0.2883,
